<a href="https://colab.research.google.com/github/donoftime2018/Mental-Health-Chatbot/blob/generateText/NLP_DialoGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
device = torch.device("cuda")
device

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small", torch_dtype=torch.float32)
tokenizer.pad_token = tokenizer.eos_token
tokenizer

In [ ]:
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-small",torch_dtype=torch.float32)
model

In [ ]:
model.to(device)

In [ ]:
dataset = pd.read_csv("normalized_context_and_response.csv")
dataset.head()

In [ ]:
contexts = dataset['context'].astype('str').values
contexts[:5]

In [ ]:
responses = dataset['response'].astype('str').values
responses[:5]

In [ ]:
def combineText(example):
    return {
        "text": f"User: {example['context']} Bot: {example['response']}"
    }

In [ ]:
def encode(example):
    return tokenizer(example['text'], truncation=True, padding=True, max_length=128)

In [ ]:
def add_labels(example):
    example['labels']=example['input_ids']
    return example

In [ ]:
datasets = Dataset.from_dict({
    "context": contexts,
    "response": responses
})
datasets

In [ ]:
datasets_split = datasets.train_test_split(test_size=0.2)
datasets_split

In [ ]:
trainSet = datasets_split['train']
trainSet

In [ ]:
testSet = datasets_split['test']
testSet

In [ ]:
trainSet = trainSet.map(combineText)
trainSet

In [ ]:
testSet = testSet.map(combineText)
testSet

In [ ]:
trainSet = trainSet.map(encode, batched=True)
trainSet

In [ ]:
testSet = testSet.map(encode, batched=True)
testSet

In [ ]:
trainSet = trainSet.map(add_labels)
trainSet

In [ ]:
testSet = testSet.map(add_labels)
testSet

In [ ]:
print(set(trainSet[0]["labels"]))

In [ ]:
trainingArgs = TrainingArguments(
    output_dir="./output",
    num_train_epochs=1,
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    eval_strategy='epoch',
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir=None,
    fp16=False,
    bf16=False,
    max_grad_norm = 1.0
)

In [ ]:
trainSet = trainSet.select(range(1100))

In [ ]:
testSet = testSet.select(range(670))

In [ ]:
inputs = tokenizer.encode(
	input(">> User: ") + " Bot: " + tokenizer.eos_token, return_tensors='pt'
).to(model.device)

outputs = model.generate(input_ids=inputs, max_length=100, max_new_tokens=100, do_sample=True, temperature=0.9, top_p=0.9, num_return_sequences=5)

In [ ]:
tokenizer.decode(outputs, skip_special_tokens=True)

In [ ]:
trainer=Trainer(
    model=model,
    args=trainingArgs,
    train_dataset= trainSet,
    eval_dataset=testSet
)

In [ ]:
trainer.evaluate(testSet)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("mental-health-dialogpt")

In [ ]:
trainer.evaluate(testSet)

In [ ]:
fineTunedModel = AutoModelForCausalLM.from_pretrained("/content/mental-health-dialogpt", torch_dtype=torch.float32)
fineTunedModel

In [ ]:
# fineTunedTokenizer = AutoTokenizer.from_pretrained("/content/mental-health-dialogpt", torch_dtype=torch.float32)
# fineTunedTokenizer.pad_token = fineTunedTokenizer.eos_token
# fineTunedTokenizer

In [ ]:
inputs = tokenizer.encode(
	input(">> User: ") + " Bot: " + tokenizer.eos_token, return_tensors='pt'
).to(fineTunedModel.device)

outputs = fineTunedModel.generate(input_ids=inputs,repetition_penalty=3.0, max_new_tokens=100, do_sample=True, temperature=0.7, top_p=0.7, top_k=50, num_return_sequences=5)

In [ ]:
tokenizer.decode(outputs, skip_special_tokens=True)